# CardioSafe AI - Notebook 06b: a learned QT/QTc estimator

**Why this notebook exists (negative result from notebook 06):** off-the-shelf wavelet delineation (neurokit2) tracked heart rate well (RR r = 0.93-0.95) but not QT: QTc agreement with 12SL was r = 0.20 with a +37 ms bias, caused by a subpopulation where the T-wave end was detected too late. A linear "correction" only collapsed predictions to the population mean (slope 0.12), which gave a flag sensitivity of 0.

**What we do instead:** train the same xResNet family to regress **QT and RR** from the 100 Hz ECG, using the **consensus of two independent algorithms** (GE 12SL and Glasgow Uni-G, from PTB-XL+) as targets, with the standard split (train 1-8, validation 9 for early stopping, test 10 untouched). 3 seeds are ensembled.

**How we judge it honestly:**
1. against a **constant predictor** (predicting the training mean), since reference QTc values are tightly clustered and a constant already looks "good" on MAE;
2. against the **agreement between 12SL and Uni-G themselves**, which shows what agreement can realistically be;
3. through the prolonged-QTc **flag** (sensitivity/specificity), where the earlier approach failed completely.

Caveat for the report: the targets are algorithm outputs, not manual cardiologist measurements. Our model learns to agree with established algorithms; it is not validated against clinical truth.

**Runtime: GPU (T4).** About 25-35 minutes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
WORK = '/content/drive/MyDrive/CardioSafe'
CKPT = WORK + '/models'; os.makedirs(CKPT, exist_ok=True)
sys.path.insert(0, WORK)

In [ ]:
%%writefile /content/drive/MyDrive/CardioSafe/cardiosafe_qt.py
"""QT / QTc measurement from 12-lead ECGs (neurokit2 DWT delineation) and agreement statistics."""
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

MEASURE_COLS = ['n_beats', 'rr_ms', 'hr', 'qt_median', 'qt_max', 'qtc_baz_median', 'qtc_baz_max',
                'qtc_fri_median', 'qtc_fri_max', 'n_leads_ok']


def _qt_one_lead(clean, rpeaks, fs):
    """Per-beat QT (ms) = T-offset minus QRS-onset on one lead; returns median over plausible beats, count."""
    import neurokit2 as nk
    _, w = nk.ecg_delineate(clean, rpeaks, sampling_rate=fs, method='dwt', show=False)
    on = np.array(w['ECG_R_Onsets'], dtype=float)
    off = np.array(w['ECG_T_Offsets'], dtype=float)
    n = min(len(on), len(off))
    qt = (off[:n] - on[:n]) / fs * 1000.0
    qt = qt[np.isfinite(qt) & (qt > 250) & (qt < 650)]
    return (float(np.median(qt)), len(qt)) if len(qt) >= 3 else (np.nan, len(qt))


def measure_qt(sig, fs=500, r_lead=1, leads=(1, 6, 10)):
    """sig: (T, 12) in mV. R-peaks from `r_lead` (default II); QT on `leads` (default II, V1... index 6 = V1, 10 = V5).
    Returns dict with MEASURE_COLS; NaN where the measurement is not reliable."""
    import neurokit2 as nk
    out = {k: np.nan for k in MEASURE_COLS}
    try:
        clean_r = nk.ecg_clean(sig[:, r_lead], sampling_rate=fs, method='neurokit')
        _, info = nk.ecg_peaks(clean_r, sampling_rate=fs, method='neurokit')
        rp = np.asarray(info['ECG_R_Peaks'])
        if len(rp) < 4:
            return out
        rr = np.diff(rp) / fs
        rr = rr[(rr > 0.3) & (rr < 2.0)]
        if len(rr) < 3:
            return out
        rr_med = float(np.median(rr))
        out.update(n_beats=len(rp), rr_ms=rr_med * 1000, hr=60 / rr_med)
        qts = []
        for l in leads:
            try:
                clean = clean_r if l == r_lead else nk.ecg_clean(sig[:, l], sampling_rate=fs, method='neurokit')
                q, _ = _qt_one_lead(clean, rp, fs)
                if np.isfinite(q):
                    qts.append(q)
            except Exception:
                continue
        out['n_leads_ok'] = len(qts)
        if qts:
            for tag, q in (('median', float(np.median(qts))), ('max', float(np.max(qts)))):
                out[f'qt_{tag}'] = q
                out[f'qtc_baz_{tag}'] = q / np.sqrt(rr_med)
                out[f'qtc_fri_{tag}'] = q / np.cbrt(rr_med)
    except Exception:
        pass
    return out


def measure_record(path, fs_expected=500, **kw):
    """Worker for parallel use: read a WFDB record (full path without extension) and measure."""
    import wfdb
    try:
        sig, fields = wfdb.rdsamp(path)
        return measure_qt(sig.astype(np.float64), fs=int(fields['fs']), **kw)
    except Exception:
        return {k: np.nan for k in MEASURE_COLS}


# ------------------------------------------------------------- agreement
def agreement(ours, ref):
    """Bland-Altman style statistics for paired measurements (ms)."""
    m = np.isfinite(ours) & np.isfinite(ref)
    a, b = np.asarray(ours)[m], np.asarray(ref)[m]
    d = a - b
    sd = d.std(ddof=1)
    return {'n': int(m.sum()), 'bias_ms': float(d.mean()), 'MAE_ms': float(np.abs(d).mean()),
            'SD_diff_ms': float(sd), 'LoA_low': float(d.mean() - 1.96 * sd), 'LoA_high': float(d.mean() + 1.96 * sd),
            'pearson_r': float(np.corrcoef(a, b)[0, 1]),
            'within_20ms': float((np.abs(d) <= 20).mean()), 'within_30ms': float((np.abs(d) <= 30).mean()),
            'within_40ms': float((np.abs(d) <= 40).mean())}


def prolonged_threshold(sex, male_ms=450, female_ms=460):
    """Commonly used sex-specific QTc limits. PTB-XL sex coding: 0 = male, 1 = female."""
    return np.where(np.asarray(sex) == 0, male_ms, female_ms).astype(float)


def classification_stats(flag_pred, flag_true):
    p, t = np.asarray(flag_pred, bool), np.asarray(flag_true, bool)
    tp, fp, fn, tn = (p & t).sum(), (p & ~t).sum(), (~p & t).sum(), (~p & ~t).sum()
    return {'sensitivity': tp / max(tp + fn, 1), 'specificity': tn / max(tn + fp, 1), 'PPV': tp / max(tp + fp, 1),
            'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn)}


In [ ]:
%%writefile /content/drive/MyDrive/CardioSafe/cardiosafe_qtnet.py
"""Regression training for the learned QT/RR estimator (targets may contain NaN; loss is masked)."""
import copy, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fn


def masked_huber(pred, y, delta=1.0):
    m = torch.isfinite(y)
    if not bool(m.any()):
        return pred.sum() * 0
    d = torch.where(m, pred - torch.nan_to_num(y), torch.zeros_like(pred))
    loss = Fn.huber_loss(d, torch.zeros_like(d), reduction='none', delta=delta)
    return (loss * m).sum() / m.sum()


@torch.no_grad()
def predict_regress(model, X, mu, sd, bs=512):
    """Returns de-standardised predictions (N, n_targets) in original units (ms)."""
    model.eval()
    out = torch.cat([model(X[i:i + bs]).float().cpu() for i in range(0, len(X), bs)]).numpy()
    return out * sd + mu


def _mae(pred, y):
    m = np.isfinite(y)
    return float(np.abs(pred[m] - y[m]).mean())


def train_regress(model, Xtr, Ytr_raw, Xva, Yva_raw, mu, sd, epochs=40, lr=2e-3, wd=1e-2, bs=128,
                  seed=0, augment=None, verbose=True):
    """Ytr_raw / Yva_raw: numpy (N, n_targets) in ms with NaN where missing. Column 0 (QT) drives early stopping."""
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    dev = Xtr.device
    model = model.to(dev)
    Ytr = torch.tensor((Ytr_raw - mu) / sd, dtype=torch.float32, device=dev)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    steps = int(np.ceil(len(Xtr) / bs)) * epochs
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.2)
    use_amp = dev.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    best, best_state, hist = 1e9, None, []
    for ep in range(epochs):
        model.train(); t0 = time.time(); tot = 0.0
        perm = torch.randperm(len(Xtr), device=dev)
        for i in range(0, len(Xtr), bs):
            idx = perm[i:i + bs]
            xb, yb = Xtr[idx], Ytr[idx]
            if augment is not None:
                xb = augment(xb)
            with torch.autocast(device_type=dev.type, enabled=use_amp):
                loss = masked_huber(model(xb).float(), yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(idx)
        pv = predict_regress(model, Xva, mu, sd)
        mae_qt = _mae(pv[:, 0], Yva_raw[:, 0])
        hist.append((ep, tot / len(Xtr), mae_qt))
        if mae_qt < best:
            best, best_state = mae_qt, copy.deepcopy(model.state_dict())
        if verbose:
            print(f'ep {ep + 1:02d}/{epochs} loss {tot / len(Xtr):.4f}  val QT MAE {mae_qt:.2f} ms  best {best:.2f}  ({time.time() - t0:.0f}s)')
    model.load_state_dict(best_state)
    return model, hist


In [ ]:
%%bash
mkdir -p /content/ptbxlplus && cd /content/ptbxlplus
for f in feature_description.csv 12sl_features.csv unig_features.csv ecgdeli_features.csv; do
  [ -s $f ] || wget -q https://physionet.org/files/ptb-xl-plus/1.0.1/features/$f
done
cd /content
[ -s ptbxl_database.csv ] || wget -q https://physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv
ls -la /content/ptbxlplus /content/ptbxl_database.csv

## Data and targets

In [ ]:
import importlib, ast, json, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import cardiosafe_common as C, cardiosafe_models as MZ, cardiosafe_qt as Q, cardiosafe_qtnet as QN
importlib.reload(Q); importlib.reload(QN)

X, Y, fold, tr, va, te, meta = C.load_data(WORK + '/ptbxl_100hz_super5.npz')
ids, sex = meta['ecg_id'], meta['sex']
PLUS = '/content/ptbxlplus'

def load_feat(name):
    df = pd.read_csv(f'{PLUS}/{name}_features.csv')
    idc = [c for c in df.columns if c.lower() in ('ecg_id', 'ecgid', 'id')]
    return df.set_index(idc[0] if idc else df.columns[0])
CAND = {'qt':      ['QT_Int_Global', 'Int_QT_Interval', 'QT_Interval', 'QTi_max'],
        'qtc_baz': ['QT_IntBazett_Global', 'QTc_Bazett'],
        'qtc_fri': ['QT_IntFridericia_Global', 'QTc_Fridericia', 'QTcFredericia'],
        'rr':      ['RR_Mean_Global', 'GM_Average_RR_Sinus', 'AvgRRInterval', 'RRi_max']}
def find(df, names):
    low = {c.lower(): c for c in df.columns}
    return next((low[n.lower()] for n in names if n.lower() in low), None)

LIM = {'qt': (250, 700), 'qtc_baz': (250, 800), 'qtc_fri': (250, 800), 'rr': (300, 2000)}
REF = {}
for alg in ['12sl', 'unig']:
    df = load_feat(alg); cols = {k: find(df, v) for k, v in CAND.items()}
    print(alg, 'matched:', cols)
    d = pd.DataFrame({k: df[c].reindex(ids).astype(float).values for k, c in cols.items() if c is not None}, index=ids)
    for k in d.columns:
        lo, hi = LIM[k]; d.loc[(d[k] < lo) | (d[k] > hi), k] = np.nan
    REF[alg] = d
def consensus(k):
    a, b = REF['12sl'][k].values, REF['unig'][k].values
    return np.where(np.isfinite(a) & np.isfinite(b), (a + b) / 2, np.where(np.isfinite(a), a, b))
REF['consensus'] = pd.DataFrame({k: consensus(k) for k in CAND}, index=ids)
Yt = REF['consensus'][['qt', 'rr']].values.astype(np.float32)
for nm, m in [('train', tr), ('val', va), ('test', te)]:
    print(f'{nm}: QT target available for {np.isfinite(Yt[m, 0]).mean() * 100:.1f}%, RR for {np.isfinite(Yt[m, 1]).mean() * 100:.1f}%')
print('QT target (train): mean %.1f sd %.1f ms' % (np.nanmean(Yt[tr, 0]), np.nanstd(Yt[tr, 0])))

## Reference points before training
(a) How well do the two established algorithms agree with each other on the test fold? (b) How well does a *constant* (training-mean) predictor do against 12SL? Our model has to beat (b) clearly and should approach (a).

In [ ]:
rows = []
for k, nm in [('qt', 'QT'), ('rr', 'RR'), ('qtc_fri', 'QTc Fridericia'), ('qtc_baz', 'QTc Bazett')]:
    a, b = REF['unig'][k].values[te], REF['12sl'][k].values[te]
    rows.append({'quantity': nm, 'comparison': 'Uni-G vs 12SL (two real algorithms)', **Q.agreement(a, b)})
    const = np.full(te.sum(), np.nanmean(REF['12sl'][k].values[tr]))
    rows.append({'quantity': nm, 'comparison': 'constant predictor vs 12SL', **Q.agreement(const, b)})
bench = pd.DataFrame(rows).set_index(['quantity', 'comparison'])
display(bench[['n', 'bias_ms', 'MAE_ms', 'SD_diff_ms', 'pearson_r', 'within_20ms', 'within_30ms']].round(3))
bench.to_csv(WORK + '/results_06b_benchmarks.csv')

## Train 3 seeds (resumable: finished seeds are loaded from Drive)

In [ ]:
dev = torch.device('cuda')
mu_x, sd_x = C.standardize_fit(X[tr])
Xtr, Xva, Xte = [C.to_tensor_ncl(X[m], mu_x, sd_x, dev) for m in (tr, va, te)]
mu_y, sd_y = np.nanmean(Yt[tr], 0), np.nanstd(Yt[tr], 0)
AUG = MZ.make_augment()
EPOCHS = 40
SEEDS = [0, 1, 2]
pv, pt, hists = {}, {}, {}
for s in SEEDS:
    net = MZ.XResNet1D(n_classes=2, width=32).to(dev)
    f = f'{CKPT}/qtnet_s{s}.pt'
    if os.path.exists(f):
        net.load_state_dict(torch.load(f, map_location=dev)); print(f'[loaded] seed {s}')
    else:
        print(f'===== seed {s} =====')
        net, hists[s] = QN.train_regress(net, Xtr, Yt[tr], Xva, Yt[va], mu_y, sd_y, epochs=EPOCHS, seed=s, augment=AUG, verbose=False)
        torch.save(net.state_dict(), f); print('   best val QT MAE: %.2f ms' % min(h[2] for h in hists[s]))
    pv[s] = QN.predict_regress(net, Xva, mu_y, sd_y); pt[s] = QN.predict_regress(net, Xte, mu_y, sd_y)
    m = np.isfinite(Yt[te, 0]); print(f'   seed {s}: test QT MAE vs consensus {np.abs(pt[s][m, 0] - Yt[te, 0][m]).mean():.2f} ms')

## Evaluation on the untouched test fold

In [ ]:
def derive(p):
    qt, rr = p[:, 0], p[:, 1]
    return {'qt': qt, 'rr': rr, 'qtc_fri': qt / np.cbrt(rr / 1000), 'qtc_baz': qt / np.sqrt(rr / 1000)}
ens_te = derive(np.mean([pt[s] for s in SEEDS], 0)); ens_va = derive(np.mean([pv[s] for s in SEEDS], 0))
seed_sd_qt = np.std([pt[s][:, 0] for s in SEEDS], 0)

rows = []
for k, nm in [('qt', 'QT'), ('rr', 'RR'), ('qtc_fri', 'QTc Fridericia'), ('qtc_baz', 'QTc Bazett')]:
    for r in ['12sl', 'unig', 'consensus']:
        if k in REF[r].columns:
            rows.append({'quantity': nm, 'comparison': f'learned ensemble vs {r}', **Q.agreement(ens_te[k], REF[r][k].values[te])})
res = pd.DataFrame(rows).set_index(['quantity', 'comparison'])
res.to_csv(WORK + '/results_06b_agreement.csv')
display(res[['n', 'bias_ms', 'MAE_ms', 'SD_diff_ms', 'LoA_low', 'LoA_high', 'pearson_r', 'within_20ms', 'within_30ms']].round(3))
print('Compare each quantity with the benchmark table above: the learned model should beat the constant predictor clearly (lower MAE, much higher r) and approach the Uni-G vs 12SL agreement.')

In [ ]:
a, b = ens_te['qtc_fri'], REF['12sl']['qtc_fri'].values[te]; mk = np.isfinite(a) & np.isfinite(b)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.8))
ax[0].scatter(b[mk], a[mk], s=4, alpha=.3); lim = [330, 620]; ax[0].plot(lim, lim, 'r--'); ax[0].set_xlim(lim); ax[0].set_ylim(lim)
ax[0].set_xlabel('12SL QTc Fridericia (ms)'); ax[0].set_ylabel('learned QTc Fridericia (ms)'); ax[0].set_title('Test fold agreement')
d = a[mk] - b[mk]; mn = (a[mk] + b[mk]) / 2
ax[1].scatter(mn, d, s=4, alpha=.3)
for yv, st in [(d.mean(), '-'), (d.mean() + 1.96 * d.std(), '--'), (d.mean() - 1.96 * d.std(), '--')]: ax[1].axhline(yv, color='r' if st == '--' else 'k', ls=st)
ax[1].set_xlabel('mean of the two (ms)'); ax[1].set_ylabel('learned - 12SL (ms)'); ax[1].set_title('Bland-Altman')
plt.tight_layout(); plt.savefig(WORK + '/fig_qtc_learned_agreement.png', dpi=150); plt.show()

## The prolonged-QTc flag (the part that failed before)
Prolonged = QTc at or above 450 ms (male) / 460 ms (female). Test fold only; natural prevalence, so predictive values are meaningful here.

In [ ]:
thr = Q.prolonged_threshold(sex[te])
rows = []
for r in ['12sl', 'unig']:
    ref = REF[r]['qtc_fri'].values[te]; v = np.isfinite(ref) & np.isfinite(ens_te['qtc_fri'])
    rows.append({'comparison': f'learned flag vs {r} flag (Fridericia)', 'n': int(v.sum()), 'ref_prolonged_%': 100 * (ref[v] >= thr[v]).mean(),
                 **Q.classification_stats(ens_te['qtc_fri'][v] >= thr[v], ref[v] >= thr[v])})
v = np.isfinite(REF['12sl']['qtc_fri'].values[te]) & np.isfinite(REF['unig']['qtc_fri'].values[te])
rows.append({'comparison': 'Uni-G flag vs 12SL flag (benchmark)', 'n': int(v.sum()), 'ref_prolonged_%': 100 * (REF['12sl']['qtc_fri'].values[te][v] >= thr[v]).mean(),
             **Q.classification_stats(REF['unig']['qtc_fri'].values[te][v] >= thr[v], REF['12sl']['qtc_fri'].values[te][v] >= thr[v])})
flag = pd.DataFrame(rows).set_index('comparison'); display(flag.round(3)); flag.to_csv(WORK + '/results_06b_flag.csv')

In [ ]:
# Agreement with PTB-XL's own long-QT annotation (folds 9 + 10; few positives, so intervals are wide)
from sklearn.metrics import roc_auc_score
db = pd.read_csv('/content/ptbxl_database.csv', index_col='ecg_id'); db['scp_codes'] = db['scp_codes'].apply(ast.literal_eval)
lab_all = np.array([('LNGQT' in db.loc[i, 'scp_codes']) if i in db.index else False for i in ids])
m910 = va | te
pred910 = np.zeros(len(ids)); pred910[va] = ens_va['qtc_fri']; pred910[te] = ens_te['qtc_fri']
rng = np.random.default_rng(0)
def auc_ci(score, lab, n=500):
    v = np.isfinite(score); s, l = score[v], lab[v]
    if l.sum() < 2 or (~l).sum() < 2: return np.nan, np.nan, np.nan          # too few positives to evaluate
    pt_ = roc_auc_score(l, s); bs = []
    for _ in range(n):
        i = rng.integers(0, len(s), len(s))
        if l[i].any() and (~l[i]).any(): bs.append(roc_auc_score(l[i], s[i]))
    return pt_, np.quantile(bs, .025), np.quantile(bs, .975)
rows = []
for nm, sc in [('learned QTc (Fridericia)', pred910), ('12SL QTc (Fridericia)', REF['12sl']['qtc_fri'].values), ('Uni-G QTc (Fridericia)', REF['unig']['qtc_fri'].values)]:
    a_, lo, hi = auc_ci(sc[m910], lab_all[m910]); rows.append({'score': nm, 'AUROC_vs_LNGQT': a_, 'ci_low': lo, 'ci_high': hi})
print('LNGQT-positive ECGs in folds 9+10:', int(lab_all[m910].sum()), 'of', int(m910.sum()))
display(pd.DataFrame(rows).set_index('score').round(3))

## Three-way QTc outcome with an honest borderline band
The band comes from **validation-fold residuals** (learned minus 12SL QTc). 'Prolonged' only if the whole band is above the threshold, 'normal' only if it is entirely below, otherwise 'borderline'. This is what feeds the "insufficient evidence" outcome of the combined report.

In [ ]:
ref_v = REF['12sl']['qtc_fri'].values[va]; vv = np.isfinite(ref_v)
resid_sd = float(np.std(ens_va['qtc_fri'][vv] - ref_v[vv], ddof=1)); band = 1.96 * resid_sd
print(f'validation residual SD {resid_sd:.1f} ms -> band +/- {band:.0f} ms')
pred = ens_te['qtc_fri']; ref = REF['12sl']['qtc_fri'].values[te]; ok = np.isfinite(pred) & np.isfinite(ref)
cat = np.where(pred - band >= thr, 'prolonged', np.where(pred + band < thr, 'normal', 'borderline'))
rows = []
for c in ['normal', 'borderline', 'prolonged']:
    mc = ok & (cat == c)
    rows.append({'outcome': c, 'n': int(mc.sum()), 'share_%': 100 * mc.sum() / ok.sum(), '12SL says prolonged_%': 100 * (ref[mc] >= thr[mc]).mean() if mc.any() else np.nan})
three = pd.DataFrame(rows).set_index('outcome'); display(three.round(2)); three.to_csv(WORK + '/results_06b_three_way.csv')
print('Good behaviour: 12SL-prolonged % is low for "normal", high for "prolonged", in between for "borderline".')

## Save the configuration for the combined module and the demo

In [ ]:
cfg = {'method': 'learned xresnet1d ensemble on 100 Hz 12-lead ECG', 'seeds': SEEDS, 'checkpoints': [f'models/qtnet_s{s}.pt' for s in SEEDS],
       'targets': 'mean of 12SL and Uni-G QT and RR (PTB-XL+)', 'qtc_formula': 'fridericia: QT / (RR/1000)**(1/3)',
       'target_mean': mu_y.tolist(), 'target_sd': sd_y.tolist(), 'input_norm_file': 'norm_stats.npz',
       'residual_sd_ms': resid_sd, 'borderline_band_ms': float(band), 'male_threshold_ms': 450, 'female_threshold_ms': 460}
json.dump(cfg, open(WORK + '/qtc_config_learned.json', 'w'), indent=1); print(json.dumps(cfg, indent=1))
pd.DataFrame({'ecg_id': ids[te], 'qt_hat': ens_te['qt'], 'rr_hat': ens_te['rr'], 'qtc_fri_hat': ens_te['qtc_fri'], 'seed_sd_qt': seed_sd_qt}).to_csv(WORK + '/qtc_test_predictions.csv', index=False)

## Next
Send me: the two benchmark rows, the agreement table, the flag table, the LNGQT AUROC table, the three-way table and `fig_qtc_learned_agreement.png`.
If the learned estimator beats the constant predictor clearly and the flag has real sensitivity, we use it. If not, I will tell you plainly and we reduce the claim to "QTc is taken from a supplied value or flagged as unavailable".